# FaceXplain — clean, reproducible re-run (ASD facial image classification + XAI)

This notebook replaces the earlier experiment notebooks with **one leakage-free pipeline**. Run it top to bottom on Colab (GPU runtime).

**What it fixes compared with the earlier notebooks**

| Earlier problem | What this notebook does |
|---|---|
| `VGG16_FINAL`: no test set; the same 20% was used for early stopping, threshold choice *and* reporting | Separate train / validation / test. Early stopping and the threshold use **validation only**; the test set is evaluated **once** at the end |
| `VGG16 (Augmentation)`: `random.shuffle` ran separately for train, val and test, so ~80% of test images were also in training | Split is created **once**, with a fixed seed, and saved to CSV |
| `alldata` (10,007 images) of unclear origin, possibly augmented copies / merged duplicates | Uses the **standard Kaggle split** (train / valid / test) and checks for **near-duplicate faces across splits**, removing them from training |
| Baseline loop evaluated the same model for every architecture (all 50%) | Each architecture is built, trained and evaluated **separately**, with its **own preprocessing** (EfficientNet previously failed because images were divided by 255) |
| Grad-CAM / LIME used a different model file and random images from all data | XAI uses **the same trained model**, on **test images only** |

**If Colab disconnects:** just click *Runtime → Run all* again. Every model/seed that already finished is saved to Drive and is loaded instead of retrained, so the run continues where it stopped.

**Runtime (T4 GPU):** roughly 15–25 min for VGG16 × 3 seeds, plus ~5–10 min per baseline model. Set `RUN_BASELINES = False` to skip baselines.

## 0. Settings — the only cell you normally need to edit

In [ ]:
# ---- Paths -------------------------------------------------------------
# Folder with the standard Kaggle split:  train/ valid/ test/ , each with autistic/ and non_autistic/
DATA_DIR = "/content/drive/MyDrive/THESIS DATA"
OUT_DIR  = "/content/drive/MyDrive/FaceXplain_clean_rerun"   # all results go to YOUR Drive (shared folder may be read-only)

# ---- Experiment settings -------------------------------------------------
SEEDS            = [42, 123, 2024]   # main model is trained once per seed -> report mean ± std
BASELINE_SEEDS   = [42]              # baselines: one seed is enough (add more if time allows)
RUN_BASELINES    = True
BASELINES        = ["VGG19", "ResNet50", "InceptionV3", "DenseNet121", "MobileNetV2", "EfficientNetB0"]

IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS_STAGE1 = 15     # frozen backbone, train head only
EPOCHS_STAGE2 = 15     # fine-tune top of backbone
PATIENCE      = 4

# The provided valid/ folder has only 100 images -> too small for reliable early stopping
# and threshold selection. If True, a stratified 10% of train/ (fixed seed) is moved into
# validation. The test/ folder is never touched.
ENLARGE_VALIDATION = True
VAL_FRACTION_FROM_TRAIN = 0.10

# Near-duplicate check: images whose 64-bit difference hash differs in <= this many bits
# are treated as the same face/photo.
DUP_HAMMING_THRESHOLD = 5
REMOVE_CROSS_SPLIT_DUPLICATES = True   # removes the TRAINING copy, so the standard test set stays intact

SPLIT_SEED = 42   # seed for the validation carve-out (kept fixed across all runs)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q lime

In [ ]:
import os, json, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve)
from sklearn.model_selection import train_test_split

os.makedirs(OUT_DIR, exist_ok=True)
FIG_DIR = os.path.join(OUT_DIR, "figures");  os.makedirs(FIG_DIR, exist_ok=True)
W_DIR   = os.path.join(OUT_DIR, "weights");  os.makedirs(W_DIR, exist_ok=True)

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

## 2. Helper functions (duplicate check, metrics, confidence intervals)

In [ ]:
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp')

def list_split(split_dir):
    # label 1 = ASD (autistic), 0 = Non-ASD (non_autistic)
    paths, labels = [], []
    for cls, lab in [("autistic", 1), ("non_autistic", 0)]:
        d = os.path.join(split_dir, cls)
        files = sorted(f for f in os.listdir(d) if f.lower().endswith(IMG_EXT))
        paths += [os.path.join(d, f) for f in files]
        labels += [lab] * len(files)
    return np.array(paths), np.array(labels)

def dhash64(path, size=8):
    # 64-bit difference hash: robust to resizing / recompression, so it finds re-saved copies
    img = Image.open(path).convert("L").resize((size + 1, size), Image.LANCZOS)
    a = np.asarray(img, dtype=np.int16)
    bits = (a[:, 1:] > a[:, :-1]).flatten()
    return np.packbits(bits).view(">u8")[0]

def hashes_for(paths):
    return np.array([dhash64(p) for p in paths], dtype=np.uint64)

def cross_duplicates(h_query, h_ref, thr):
    # for each query hash, the min Hamming distance to any reference hash and its index
    q = h_query.view(np.uint8).reshape(-1, 8)
    r = h_ref.view(np.uint8).reshape(-1, 8)
    best_d = np.full(len(q), 65, dtype=np.int32); best_i = np.full(len(q), -1)
    for start in range(0, len(r), 512):
        x = np.bitwise_xor(q[:, None, :], r[None, start:start + 512, :])
        d = np.unpackbits(x, axis=2).sum(axis=2)
        i = d.argmin(axis=1); m = d[np.arange(len(q)), i]
        upd = m < best_d
        best_d[upd] = m[upd]; best_i[upd] = i[upd] + start
    return best_d, best_i, best_d <= thr

def youden_threshold(y_true, y_prob):
    fpr, tpr, thr = roc_curve(y_true, y_prob)
    j = tpr - fpr
    t = float(thr[np.argmax(j)])
    return float(np.clip(t, 1e-6, 1 - 1e-6))

def metrics_at(y_true, y_prob, thr):
    y_pred = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": thr,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    }

def bootstrap_ci(y_true, y_prob, thr, n=2000, seed=0):
    # 95% percentile bootstrap CI for accuracy and AUC on the test set
    rng = np.random.default_rng(seed)
    accs, aucs = [], []
    idx = np.arange(len(y_true))
    for _ in range(n):
        s = rng.choice(idx, len(idx), replace=True)
        if len(np.unique(y_true[s])) < 2:
            continue
        accs.append(accuracy_score(y_true[s], (y_prob[s] >= thr).astype(int)))
        aucs.append(roc_auc_score(y_true[s], y_prob[s]))
    return (np.percentile(accs, [2.5, 97.5]).tolist(), np.percentile(aucs, [2.5, 97.5]).tolist())

## 3. Build the split once, check for duplicate faces, save the split

In [ ]:
tr_p, tr_y = list_split(os.path.join(DATA_DIR, "train"))
va_p, va_y = list_split(os.path.join(DATA_DIR, "valid"))
te_p, te_y = list_split(os.path.join(DATA_DIR, "test"))
print(f"Original split  train {len(tr_p)} | valid {len(va_p)} | test {len(te_p)}")

# ---- near-duplicate check across splits -----------------------------------
t0 = time.time()
h_tr, h_va, h_te = hashes_for(tr_p), hashes_for(va_p), hashes_for(te_p)
print(f"Hashed {len(h_tr)+len(h_va)+len(h_te)} images in {time.time()-t0:.0f}s")

d_te, i_te, dup_te = cross_duplicates(h_te, h_tr, DUP_HAMMING_THRESHOLD)
d_va, i_va, dup_va = cross_duplicates(h_va, h_tr, DUP_HAMMING_THRESHOLD)
d_tv, _,    dup_tv = cross_duplicates(h_te, h_va, DUP_HAMMING_THRESHOLD)
print(f"Test images with a near-duplicate in TRAIN : {dup_te.sum()}")
print(f"Valid images with a near-duplicate in TRAIN: {dup_va.sum()}")
print(f"Test images with a near-duplicate in VALID : {dup_tv.sum()}")

dup_rows = [{"query_split": "test", "query": te_p[k], "train_match": tr_p[i_te[k]], "hamming": int(d_te[k])}
            for k in np.where(dup_te)[0]]
dup_rows += [{"query_split": "valid", "query": va_p[k], "train_match": tr_p[i_va[k]], "hamming": int(d_va[k])}
             for k in np.where(dup_va)[0]]
dup_df = pd.DataFrame(dup_rows)
dup_df.to_csv(os.path.join(OUT_DIR, "cross_split_duplicates.csv"), index=False)

# Show a few pairs so you can confirm they really are the same photo
if len(dup_df):
    show = dup_df.head(6)
    fig, ax = plt.subplots(2, len(show), figsize=(2.6 * len(show), 5.4))
    ax = np.array(ax).reshape(2, -1)
    for c, (_, r) in enumerate(show.iterrows()):
        ax[0, c].imshow(Image.open(r["query"]).convert("RGB")); ax[0, c].set_title(f'{r["query_split"]} (d={r["hamming"]})', fontsize=8)
        ax[1, c].imshow(Image.open(r["train_match"]).convert("RGB")); ax[1, c].set_title("train match", fontsize=8)
        ax[0, c].axis("off"); ax[1, c].axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "duplicate_examples.png"), dpi=150); plt.show()

# ---- remove the TRAINING copies of cross-split duplicates ----------------------
if REMOVE_CROSS_SPLIT_DUPLICATES:
    drop = set(i_te[dup_te].tolist()) | set(i_va[dup_va].tolist())
    keep = np.array([k not in drop for k in range(len(tr_p))])
    tr_p, tr_y = tr_p[keep], tr_y[keep]
    print(f"Removed {len(drop)} duplicate images from TRAIN")

# Exact duplicates inside train are harmless for evaluation, so they are kept.

# ---- enlarge validation from train (stratified, fixed seed) ------------------
if ENLARGE_VALIDATION:
    tr_p, extra_p, tr_y, extra_y = train_test_split(
        tr_p, tr_y, test_size=VAL_FRACTION_FROM_TRAIN, stratify=tr_y, random_state=SPLIT_SEED)
    va_p, va_y = np.concatenate([va_p, extra_p]), np.concatenate([va_y, extra_y])

split_df = pd.concat([
    pd.DataFrame({"split": "train", "path": tr_p, "label": tr_y}),
    pd.DataFrame({"split": "val",   "path": va_p, "label": va_y}),
    pd.DataFrame({"split": "test",  "path": te_p, "label": te_y}),
])
split_df.to_csv(os.path.join(OUT_DIR, "final_split.csv"), index=False)

counts = (split_df.assign(cls=split_df.label.map({1: "ASD", 0: "Non-ASD"}))
          .pivot_table(index="split", columns="cls", values="path", aggfunc="count")
          .reindex(["train", "val", "test"]))
counts["Total"] = counts.sum(axis=1)
print("\nFinal split used for ALL experiments:"); print(counts)
assert not (set(tr_p) & set(te_p)) and not (set(tr_p) & set(va_p)) and not (set(va_p) & set(te_p)), "Split overlap!"

## 4. Data pipeline
Images are kept in the 0–255 range; each model applies its **own** ImageNet preprocessing inside the network. Augmentation is applied **only during training** (Keras augmentation layers are inactive at inference).

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_img(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), tf.cast(label, tf.float32)

def make_ds(paths, labels, shuffle, seed):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels)).map(load_img, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(len(paths), seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def predict_probs(model, paths):
    ds = make_ds(paths, np.zeros(len(paths)), shuffle=False, seed=0)
    return model.predict(ds, verbose=0).ravel()

## 5. Model definitions
All models share the same head and training protocol, so the comparison is fair.

In [ ]:
A = keras.applications
BACKBONES = {
    #  name           constructor          preprocessing                         last conv (Grad-CAM)   layers unfrozen in stage 2
    "VGG16":         (A.VGG16,          A.vgg16.preprocess_input,           "block5_conv3",          "block5"),
    "VGG19":         (A.VGG19,          A.vgg19.preprocess_input,           "block5_conv4",          "block5"),
    "ResNet50":      (A.ResNet50,       A.resnet50.preprocess_input,        "conv5_block3_out",      30),
    "InceptionV3":   (A.InceptionV3,    A.inception_v3.preprocess_input,    "mixed10",               40),
    "DenseNet121":   (A.DenseNet121,    A.densenet.preprocess_input,        "relu",                  40),
    "MobileNetV2":   (A.MobileNetV2,    A.mobilenet_v2.preprocess_input,    "out_relu",              30),
    "EfficientNetB0":(A.EfficientNetB0, A.efficientnet.preprocess_input,    "top_activation",        30),
}

def build_model(name):
    ctor, prep, _, _ = BACKBONES[name]
    base = ctor(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
        layers.RandomBrightness(0.1),     # value_range defaults to (0, 255), matching our inputs
    ], name="augmentation")
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="image")
    x = aug(inp)
    x = prep(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Dense(128, activation="relu", name="head_dense")(x)
    x = layers.Dropout(0.5, name="head_dropout")(x)
    out = layers.Dense(1, activation="sigmoid", name="head_out")(x)
    return keras.Model(inp, out, name=f"{name}_asd"), base

def unfreeze_top(base, rule):
    base.trainable = True
    for l in base.layers:
        if isinstance(rule, str):
            l.trainable = l.name.startswith(rule)
        else:
            l.trainable = False
    if not isinstance(rule, str):
        for l in base.layers[-rule:]:
            if not isinstance(l, layers.BatchNormalization):   # keep BN statistics frozen
                l.trainable = True

def compile_model(model, lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.AUC(name="auc")])

## 6. Training + evaluation protocol (identical for every model)
1. Stage 1: backbone frozen, head trained.
2. Stage 2: top of backbone unfrozen, lower learning rate.
3. Early stopping on **validation AUC**.
4. Decision threshold chosen on **validation** (Youden's J).
5. **Test set evaluated once**, at that threshold (results at 0.5 are also saved).

In [ ]:
def run_experiment(name, seed):
    # RESUME: if this model+seed already finished in an earlier (disconnected) session, reuse it
    tag = f"{name}_seed{seed}"
    w_path = os.path.join(W_DIR, f"{tag}.weights.h5")
    r_path = os.path.join(OUT_DIR, "runs", f"{tag}.json")
    if os.path.exists(w_path) and os.path.exists(r_path):
        with open(r_path) as f:
            saved = json.load(f)
        model, base = build_model(name)
        unfreeze_top(base, BACKBONES[name][3])
        model.load_weights(w_path)
        print(f">>> {tag}: already done earlier - loaded saved result (test acc {saved['row']['accuracy']:.4f})")
        return model, base, saved["row"], saved["hist"]

    set_seed(seed)
    model, base = build_model(name)
    train_ds = make_ds(tr_p, tr_y, shuffle=True, seed=seed)
    val_ds   = make_ds(va_p, va_y, shuffle=False, seed=seed)
    es = lambda: keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=PATIENCE,
                                               restore_best_weights=True)
    t0 = time.time()
    compile_model(model, 1e-4)
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE1, callbacks=[es()], verbose=2)
    unfreeze_top(base, BACKBONES[name][3])
    compile_model(model, 1e-5)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE2, callbacks=[es()], verbose=2)
    train_min = (time.time() - t0) / 60

    # threshold from VALIDATION only
    p_val = predict_probs(model, va_p)
    thr = youden_threshold(va_y, p_val)

    # TEST evaluated once
    p_test = predict_probs(model, te_p)
    m = metrics_at(te_y, p_test, thr)
    m05 = metrics_at(te_y, p_test, 0.5)
    (acc_lo, acc_hi), (auc_lo, auc_hi) = bootstrap_ci(te_y, p_test, thr, seed=seed)

    row = {"model": name, "seed": seed, "epochs_stage1": len(h1.history["loss"]),
           "epochs_stage2": len(h2.history["loss"]), "train_minutes": round(train_min, 1),
           "val_auc": roc_auc_score(va_y, p_val), **m,
           "accuracy_at_0.5": m05["accuracy"], "f1_at_0.5": m05["f1"],
           "acc_CI95_low": acc_lo, "acc_CI95_high": acc_hi, "auc_CI95_low": auc_lo, "auc_CI95_high": auc_hi,
           "params": model.count_params()}
    model.save_weights(w_path)
    hist = {k: [float(v) for v in h1.history[k] + h2.history[k]] for k in h1.history}
    os.makedirs(os.path.dirname(r_path), exist_ok=True)
    with open(r_path, "w") as f:
        json.dump({"row": {k: (v.item() if hasattr(v, "item") else v) for k, v in row.items()},
                   "hist": hist}, f, indent=1)
    pd.DataFrame({"path": te_p, "y_true": te_y, "p_asd": p_test,
                  "y_pred": (p_test >= thr).astype(int)}).to_csv(
        os.path.join(OUT_DIR, f"test_predictions_{name}_seed{seed}.csv"), index=False)
    print(f"\n>>> {name} seed {seed}: test acc {m['accuracy']:.4f} | AUC {m['auc']:.4f} | thr {thr:.3f}\n")
    return model, base, row, hist

## 7. Main model: VGG16 over several seeds

In [ ]:
results, histories, vgg_models = [], {}, {}
for s in SEEDS:
    mdl, base, row, hist = run_experiment("VGG16", s)
    results.append(row); histories[("VGG16", s)] = hist; vgg_models[s] = (mdl, base)

vgg_df = pd.DataFrame(results)
show_cols = ["seed", "accuracy", "precision", "recall_sensitivity", "specificity", "f1", "auc", "threshold"]
print(vgg_df[show_cols].round(4).to_string(index=False))
summary = vgg_df[show_cols[1:-1]].agg(["mean", "std"]).T
print("\nVGG16 test performance (mean ± std over seeds):")
for k, r in summary.iterrows():
    print(f"  {k:<20} {r['mean']*100:6.2f} ± {r['std']*100:.2f} %")

In [ ]:
# Training curves for the first seed
h = histories[("VGG16", SEEDS[0])]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["accuracy"], label="train"); ax[0].plot(h["val_accuracy"], label="validation"); ax[0].set_title("Accuracy")
ax[1].plot(h["loss"], label="train");     ax[1].plot(h["val_loss"], label="validation");     ax[1].set_title("Loss")
s1 = int(vgg_df.loc[vgg_df.seed == SEEDS[0], "epochs_stage1"].iloc[0])
for a in ax:
    a.axvline(s1 - 0.5, ls="--", c="grey"); a.set_xlabel("epoch"); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "vgg16_training_curves.png"), dpi=200); plt.show()

# Confusion matrix + ROC for the first seed
mdl = vgg_models[SEEDS[0]][0]
r0 = vgg_df[vgg_df.seed == SEEDS[0]].iloc[0]
p_test = predict_probs(mdl, te_p)
cm = confusion_matrix(te_y, (p_test >= r0.threshold).astype(int), labels=[0, 1])
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    ax[0].text(j, i, v, ha="center", va="center", fontsize=14)
ax[0].set_xticks([0, 1]); ax[0].set_yticks([0, 1])
ax[0].set_xticklabels(["Non-ASD", "ASD"]); ax[0].set_yticklabels(["Non-ASD", "ASD"])
ax[0].set_xlabel("Predicted"); ax[0].set_ylabel("True"); ax[0].set_title("Test confusion matrix")
fpr, tpr, _ = roc_curve(te_y, p_test)
ax[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(te_y, p_test):.3f}"); ax[1].plot([0, 1], [0, 1], "--", c="grey")
ax[1].set_xlabel("False positive rate"); ax[1].set_ylabel("True positive rate"); ax[1].set_title("Test ROC"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "vgg16_confusion_roc.png"), dpi=200); plt.show()

## 8. Baselines (same split, same protocol)

In [ ]:
base_rows = []
if RUN_BASELINES:
    for name in BASELINES:
        for s in BASELINE_SEEDS:
            mdl, _, row, hist = run_experiment(name, s)
            base_rows.append(row); histories[(name, s)] = hist
            del mdl; keras.backend.clear_session()

all_df = pd.concat([vgg_df, pd.DataFrame(base_rows)], ignore_index=True)
cmp = (all_df.groupby("model")[["accuracy", "precision", "recall_sensitivity", "specificity", "f1", "auc"]]
       .mean().sort_values("auc", ascending=False))
print("Test-set comparison (mean over the seeds run for each model):")
print((cmp * 100).round(2).to_string())

## 9. Explainability on the SAME model, TEST images only

In [ ]:
XAI_SEED = SEEDS[0]
xai_model, xai_base = vgg_models[XAI_SEED]
xai_thr = float(vgg_df.loc[vgg_df.seed == XAI_SEED, "threshold"].iloc[0])
prep = BACKBONES["VGG16"][1]
conv_model = keras.Model(xai_base.inputs, [xai_base.get_layer("block5_conv3").output, xai_base.output])
head = [xai_model.get_layer(n) for n in ["gap", "head_bn", "head_dense", "head_dropout", "head_out"]]

def load_raw(path):
    return np.asarray(Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32)

def gradcam(img_raw):
    # Grad-CAM w.r.t. the ASD probability; works through the nested backbone explicitly
    x = prep(tf.convert_to_tensor(img_raw[None].copy()))
    with tf.GradientTape() as tape:
        conv, feat = conv_model(x, training=False)
        h = feat
        for l in head:
            h = l(h, training=False)
        score = h[:, 0]
    g = tape.gradient(score, conv)[0]
    w = tf.reduce_mean(g, axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return np.asarray(Image.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)) / 255.0, float(score[0])

# sanity check: Grad-CAM path must give the same probability as the model itself
_p_model = float(xai_model.predict(load_raw(te_p[0])[None], verbose=0)[0, 0])
_, _p_cam = gradcam(load_raw(te_p[0]))
assert abs(_p_model - _p_cam) < 1e-3, (_p_model, _p_cam)
print("Grad-CAM path verified against model output.")

p_xai = predict_probs(xai_model, te_p)
pred_xai = (p_xai >= xai_thr).astype(int)
rng = np.random.default_rng(XAI_SEED)
def pick(mask, k):
    idx = np.where(mask)[0]
    return rng.choice(idx, min(k, len(idx)), replace=False) if len(idx) else []
chosen = ([("ASD, correct", i) for i in pick((te_y == 1) & (pred_xai == 1), 3)] +
          [("Non-ASD, correct", i) for i in pick((te_y == 0) & (pred_xai == 0), 3)] +
          [("misclassified", i) for i in pick(te_y != pred_xai, 2)])

fig, ax = plt.subplots(len(chosen), 3, figsize=(9, 3 * len(chosen)))
for r, (lab, i) in enumerate(chosen):
    img = load_raw(te_p[i]); cam, p = gradcam(img)
    ax[r, 0].imshow(img.astype(np.uint8)); ax[r, 0].set_title(f"{lab}\nP(ASD)={p:.2f}", fontsize=9)
    ax[r, 1].imshow(cam, cmap="jet"); ax[r, 1].set_title("Grad-CAM", fontsize=9)
    ax[r, 2].imshow(img.astype(np.uint8)); ax[r, 2].imshow(cam, cmap="jet", alpha=0.4); ax[r, 2].set_title("Overlay", fontsize=9)
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "gradcam_test_examples.png"), dpi=200); plt.show()

# Class-average Grad-CAM over the whole test set
avg = {}
for lab, name in [(1, "ASD"), (0, "Non-ASD")]:
    cams = [gradcam(load_raw(p))[0] for p in te_p[te_y == lab]]
    avg[name] = np.mean(cams, axis=0)
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(avg["ASD"], cmap="jet"); ax[0].set_title("Mean Grad-CAM: ASD (test)")
ax[1].imshow(avg["Non-ASD"], cmap="jet"); ax[1].set_title("Mean Grad-CAM: Non-ASD (test)")
im = ax[2].imshow(avg["ASD"] - avg["Non-ASD"], cmap="bwr"); ax[2].set_title("Difference (ASD − Non-ASD)")
plt.colorbar(im, ax=ax[2], fraction=0.046)
for a in ax: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "gradcam_class_average_test.png"), dpi=200); plt.show()

In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries

explainer = lime_image.LimeImageExplainer(random_state=XAI_SEED)
def lime_predict(batch):
    p = xai_model.predict(batch.astype(np.float32), verbose=0).ravel()   # model expects 0-255 input
    return np.stack([1 - p, p], axis=1)

lime_set = ([c for c in chosen if c[0] == "ASD, correct"][:2] +
            [c for c in chosen if c[0] == "Non-ASD, correct"][:2] +
            [c for c in chosen if c[0] == "misclassified"][:1])
fig, ax = plt.subplots(len(lime_set), 2, figsize=(6.5, 3.2 * len(lime_set)))
for r, (lab, i) in enumerate(lime_set):
    img = load_raw(te_p[i])
    exp = explainer.explain_instance(img.astype(np.double), lime_predict, top_labels=2,
                                     hide_color=0, num_samples=1000, random_seed=XAI_SEED)
    cls = int(lime_predict(img[None])[0, 1] >= xai_thr)
    tmp, mask = exp.get_image_and_mask(cls, positive_only=False, num_features=6, hide_rest=False)
    ax[r, 0].imshow(img.astype(np.uint8)); ax[r, 0].set_title(lab, fontsize=9)
    ax[r, 1].imshow(mark_boundaries(tmp / 255.0, mask)); ax[r, 1].set_title("LIME (green = supports, red = against)", fontsize=8)
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(FIG_DIR, "lime_test_examples.png"), dpi=200); plt.show()

## 10. Save everything and print text for the paper

In [ ]:
config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, bool))}
with open(os.path.join(OUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

with pd.ExcelWriter(os.path.join(OUT_DIR, "clean_results.xlsx")) as xw:
    counts.to_excel(xw, sheet_name="Split")
    vgg_df.to_excel(xw, sheet_name="VGG16_per_seed", index=False)
    summary.to_excel(xw, sheet_name="VGG16_mean_std")
    all_df.to_excel(xw, sheet_name="All_runs", index=False)
    cmp.to_excel(xw, sheet_name="Model_comparison")
    (dup_df if len(dup_df) else pd.DataFrame({"note": ["no cross-split duplicates"]})).to_excel(
        xw, sheet_name="Duplicates", index=False)

m = summary["mean"]; sd = summary["std"]
r0 = vgg_df.iloc[0]
print(f'''
=== TEXT FOR THE PAPER (check before pasting) ===
Data: standard Kaggle split; {int(dup_te.sum())} test and {int(dup_va.sum())} validation images had
near-duplicates in the training set (64-bit dHash, Hamming <= {DUP_HAMMING_THRESHOLD}); the training copies
were removed. Final split: train {len(tr_p)}, validation {len(va_p)}, test {len(te_p)}.

Protocol: early stopping on validation AUC; decision threshold selected on the validation set
(Youden's J); the test set was evaluated once. Results are mean ± std over {len(SEEDS)} seeds.

VGG16 test results: accuracy {m.accuracy*100:.2f} ± {sd.accuracy*100:.2f}%, precision {m.precision*100:.2f} ± {sd.precision*100:.2f}%,
recall {m.recall_sensitivity*100:.2f} ± {sd.recall_sensitivity*100:.2f}%, specificity {m.specificity*100:.2f} ± {sd.specificity*100:.2f}%,
F1 {m.f1*100:.2f} ± {sd.f1*100:.2f}%, AUC {m.auc:.3f} ± {sd.auc:.3f}.
Seed {int(r0.seed)}: 95% bootstrap CI for accuracy [{r0.acc_CI95_low*100:.1f}, {r0.acc_CI95_high*100:.1f}]%, AUC [{r0.auc_CI95_low:.3f}, {r0.auc_CI95_high:.3f}].
''')
print("All outputs saved in:", OUT_DIR)